<a href="https://colab.research.google.com/github/arman-hossain45/Datathon_Contest/blob/main/final_linear_regression_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Linear regression over all pipe line code in details

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression,ElasticNet,SGDRegressor,Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor,AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from xgboost import XGBRegressor

!pip install catboost
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor,StackingRegressor

import warnings
warnings.filterwarnings('ignore')


train_df=pd.read_csv("dataset.csv")
test_df = pd.read_csv("test.csv")
# keep a copy of the test IDs for submission
test_ids = test_df['PID'].copy()
test_df=test_df.drop(columns=['PID'])

train_df.describe().T
train_df.info()
train_df.shape
train_df.isnull().sum()
#এইখানে সবগুলা কলামের null value দেখাচ্ছে না। এখন যেসব কলামে null value আছে তা দেখার জন্য আমরা নিচের কোডটি লিখবো:
null_values = train_df.isnull().sum()
null_values[null_values > 0]


#এইখানে যাদের missing value অনেক, যেসব কলামের ওই কলামগুলো আমরা delete করে দেব।
#একটা কলাম ডিলিট করতে চাইলে এই কোড লিখবো: df.drop("Car_ID", axis=1, inplace=True)
train_df.drop(
    columns=['Misc Feature', 'Pool QC', 'Alley','Mas Vnr Type','Fireplace Qu'],
    inplace=True
)

#Duplicate value আছে কিনা তা আমরা চেক করলাম।

train_df.duplicated().sum()
train_df.drop_duplicates(inplace=True)


num=train_df.select_dtypes(include=['int64','float64'])
cat = train_df.select_dtypes(include=['object'])

# see correlation matrics in this pipeline
corr=num.corr()
sns.heatmap(corr,cmap='coolwarm',center=0,annot=False,linewidths=.5)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()
# target ar sathe correlation dekbo
target_corr=(train_df.corr(numeric_only=True)['SalePrice'].sort_values(ascending=False))
target_corr
#ক্যাটেগরিকাল কলাম যদি অনেক থাকে, তাহলে লেবেল এনকোডার দেখার জন্য এই কোডটা দরকার:

for col in cat:
    print(f"\nColumn: {col}")
    print("Unique count:", train_df[col].nunique())
    print("Values:", train_df[col].unique())

plt.figure(figsize=(15,15))
train_df[num.columns].hist(bins=20,figsize=(15,15))
plt.title("Numerical data distribution in details ")
plt.tight_layout()
plt.show()


# see all the data kde to decide the missing value handeling terms
for col in num.columns:
  plt.figure(figsize=(8,6))
  sns.histplot(train_df[col],kde=True,bins=20)
  plt.title(f"{col}")
  plt.tight_layout()
  plt.show()



plt.figure(figsize=(15,15))
train_df[num.columns].boxplot()
plt.tight_layout()
plt.show()



def handle_outlier(df, col):

    df = df.copy()

    # Calculate Q1 and Q3
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    # Calculate IQR
    IQR = q3 - q1

    # Calculate lower and upper bounds
    lower = q1 - 1.5 * IQR
    upper = q3 + 1.5 * IQR

    # See number of outliers
    outlier = df[(df[col] < lower) | (df[col] > upper)]
    print(f"Number of outliers in {col}: {len(outlier)}")

    # Handle outliers using clipping
    df[col] = df[col].clip(lower=lower, upper=upper)

    return df


train_df = handle_outlier(train_df, 'Lot Frontage')
train_df = handle_outlier(train_df, 'Lot Area')
train_df = handle_outlier(train_df, 'Wood Deck SF')

x=train_df.drop(columns=['SalePrice'])
y=train_df['SalePrice']


x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
numerical = x.select_dtypes(include=['int64','float64'])
categorical = x.select_dtypes(include=['object'])

# crate the overall pipeline

num_trans = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
cat_trans =Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='most_frequent')),
        ("onehot",OneHotEncoder(handle_unknown='ignore'))
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ('num',num_trans,numerical.columns),
        ('cat',cat_trans,categorical.columns)

    ]
)


# base model identify
lr=LinearRegression()
Elas=ElasticNet(alpha=0.1)
Sgd=SGDRegressor(loss='squared_error',max_iter=1000,alpha=0.001,random_state=42,eta0=0.01)
svr=SVR(max_iter=100,gamma=0.1,kernel='rbf',C=0.6,degree=3)
Rg=RandomForestRegressor(n_estimators=100,criterion='squared_error',min_samples_leaf=2,min_samples_split=3,max_features='sqrt',random_state=42)
Gb=GradientBoostingRegressor(n_estimators=100,loss='absolute_error',subsample=1,min_samples_leaf=3,max_features='sqrt',random_state=42)
Ada=AdaBoostRegressor(n_estimators=156,loss='linear',random_state=42)
Dtr=DecisionTreeRegressor(max_depth=3,min_samples_leaf=3,min_samples_split=3,random_state=42,criterion='absolute_error')
Knr=KNeighborsRegressor(n_neighbors=17,weights='distance',metric='minkowski')
XG=XGBRegressor(n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1)
cat_boost=CatBoostRegressor()
lightgbm=LGBMRegressor()


stacking=StackingRegressor(
    estimators=[
         ('lr',lr),
         ('elas',Elas),
         ('Sgd',Sgd),
         ("svr",svr),
         ('Rg',Rg),
         ("GB",Gb),
         ("Ada",Ada),
         ("Dtr",Dtr),
         ('Knr',Knr),
         ('XG',XG),
         ('cat_boost',cat_boost),
         ('lightgbm',lightgbm)

    ],
   final_estimator=Ridge()
)


voting=VotingRegressor(
    estimators=[
         ('lr',lr),
         ('elas',Elas),
         ('Sgd',Sgd),
         ("svr",svr),
         ('Rg',Rg),
         ("GB",Gb),
         ("Ada",Ada),
         ("Dtr",Dtr),
         ('Knr',Knr),
         ('XG',XG),
         ('cat_boost',cat_boost),
         ('lightgbm',lightgbm)

    ]

)


model_to_train={
    'Logistic Regression' : lr,
    'Elas':Elas,
    "SGDRegressor":Sgd,
    "SVR":svr,
    "RandomForestRegressor":Rg,
    "GradientBoostingRegressor":Gb,
    "AdaBoostRegressor":Ada,
    'DecisionTreeRegressor':Dtr,
    "KNeighborsRegressor":Knr,
    "XGBRegressor":XG,
    "StackingRegressor":stacking,
    "VotingRegressor":voting,
    "CatBoostRegressor":cat_boost,
    "LGBMRegressor":lightgbm


}


from sklearn.metrics import accuracy_score,r2_score
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# Trainning pipe line
result=[]
for model_name,model in model_to_train.items():
    pipe=Pipeline(
        [("preprocessor",preprocessor),
        ("model",model)
        ]
    )
    pipe.fit(x_train,y_train)
    y_pred=pipe.predict(x_test)
    accuracy=r2_score(y_test,y_pred)
    result.append(
        {
            "model":model,
            'accuracy':accuracy
        }
    )


res=pd.DataFrame(result).sort_values(by='accuracy',ascending=False)
res

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np
import pandas as pd

# Training Pipeline
result = []

for model_name, model in model_to_train.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(x_train, y_train)

    y_pred = pipe.predict(x_test)

    # Regression Metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    result.append({
        "Model": model_name,
        "R2 Score": r2,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse
    })

# Convert to DataFrame
res = pd.DataFrame(result).sort_values(
    by="R2 Score",
    ascending=False
)

res


xg_pipeline= Pipeline(
    [
        ('preprocessor',preprocessor),
        ('model',XG)
    ]
)



# Cross validation
cv_result=cross_val_score(xg_pipeline,x_train,y_train,cv=5,scoring='r2')

cv_acc=cv_result.mean()
cv_std=cv_result.std()

cv_acc,cv_std,cv_result



param_grid = {
    "model__n_estimators": [ 900, 1000, 1100],
    "model__learning_rate": [0.04, 0.1, 0.2],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_child_weight": [1, 2, 3],
    "model__subsample": [0.4, 0.5, 0.8],
    "model__colsample_bytree": [0.5, 0.8, 1.0],
    "model__reg_alpha": [0.1, 0.2, 0.5],
    "model__reg_lambda": [1, 2, 4]
}

grid=GridSearchCV(
    estimator=xg_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1

)

grid.fit(x_train,y_train)
print(grid.best_params_)
print(grid.best_score_)

model = grid.best_estimator_

feature_names = model.named_steps["preprocessor"].get_feature_names_out()

importance = model.named_steps["model"].feature_importances_

importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance
    })
    .sort_values("Importance", ascending=False)
)

print(importance.head(30))

preds=model.predict(test_df)
submission=pd.Dataframe(
    {
        'PID':test_ids,
        'SalePrice':preds
    }
)
submission.to_csv('submission.csv',index=False)




























































classification pipeline code

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV,StratifiedKFold
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from xgboost import XGBClassifier

!pip install catboost
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier,StackingClassifier

import warnings
warnings.filterwarnings('ignore')


train_df=pd.read_csv("dataset.csv")
test_df = pd.read_csv("test.csv")
# keep a copy of the test IDs for submission
test_ids = test_df['PID'].copy()
test_df=test_df.drop(columns=['PID'])

train_df.describe().T
train_df.info()
train_df.shape
train_df.isnull().sum()
#এইখানে সবগুলা কলামের null value দেখাচ্ছে না। এখন যেসব কলামে null value আছে তা দেখার জন্য আমরা নিচের কোডটি লিখবো:
null_values = train_df.isnull().sum()
null_values[null_values > 0]


#এইখানে যাদের missing value অনেক, যেসব কলামের ওই কলামগুলো আমরা delete করে দেব।
#একটা কলাম ডিলিট করতে চাইলে এই কোড লিখবো: df.drop("Car_ID", axis=1, inplace=True)
train_df.drop(
    columns=['Misc Feature', 'Pool QC', 'Alley','Mas Vnr Type','Fireplace Qu'],
    inplace=True
)

#Duplicate value আছে কিনা তা আমরা চেক করলাম।

train_df.duplicated().sum()
train_df.drop_duplicates(inplace=True)


num=train_df.select_dtypes(include=['int64','float64'])
cat = train_df.select_dtypes(include=['object'])

# see correlation matrics in this pipeline
corr=num.corr()
sns.heatmap(corr,cmap='coolwarm',center=0,annot=False,linewidths=.5)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

# target column er type check kore nichi - classification hole target categorical o hote pare, numeric label o hote pare
TARGET_COL = 'SalePrice'

if TARGET_COL in num.columns:
    # target numeric thakle o classification er jonno label encode kora hocche (class hisebe treat kora hobe)
    target_corr=(train_df.corr(numeric_only=True)[TARGET_COL].sort_values(ascending=False))
    target_corr
else:
    print(f"Target column '{TARGET_COL}' is categorical, skipping numeric correlation with target.")

#ক্যাটেগরিকাল কলাম যদি অনেক থাকে, তাহলে লেবেল এনকোডার দেখার জন্য এই কোডটা দরকার:

for col in cat:
    print(f"\nColumn: {col}")
    print("Unique count:", train_df[col].nunique())
    print("Values:", train_df[col].unique())

plt.figure(figsize=(15,15))
train_df[num.columns].hist(bins=20,figsize=(15,15))
plt.title("Numerical data distribution in details ")
plt.tight_layout()
plt.show()


# see all the data kde to decide the missing value handeling terms
for col in num.columns:
  plt.figure(figsize=(8,6))
  sns.histplot(train_df[col],kde=True,bins=20)
  plt.title(f"{col}")
  plt.tight_layout()
  plt.show()



plt.figure(figsize=(15,15))
train_df[num.columns].boxplot()
plt.tight_layout()
plt.show()



def handle_outlier(df, col):

    df = df.copy()

    # Calculate Q1 and Q3
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    # Calculate IQR
    IQR = q3 - q1

    # Calculate lower and upper bounds
    lower = q1 - 1.5 * IQR
    upper = q3 + 1.5 * IQR

    # See number of outliers
    outlier = df[(df[col] < lower) | (df[col] > upper)]
    print(f"Number of outliers in {col}: {len(outlier)}")

    # Handle outliers using clipping
    df[col] = df[col].clip(lower=lower, upper=upper)

    return df


train_df = handle_outlier(train_df, 'Lot Frontage')
train_df = handle_outlier(train_df, 'Lot Area')
train_df = handle_outlier(train_df, 'Wood Deck SF')

x=train_df.drop(columns=[TARGET_COL])
y=train_df[TARGET_COL]

# classification er jonno target ta categorical/label form e niye asha hocche
# target already categorical (object) thakle সরাসরি encode হবে,
# আর numeric thakle o eita class label hisebe treat kore encode kora hocche
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)


x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
numerical = x.select_dtypes(include=['int64','float64'])
categorical = x.select_dtypes(include=['object'])

# crate the overall pipeline

num_trans = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
cat_trans =Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='most_frequent')),
        ("onehot",OneHotEncoder(handle_unknown='ignore'))
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ('num',num_trans,numerical.columns),
        ('cat',cat_trans,categorical.columns)

    ]
)


# base model identify
lr=LogisticRegression(max_iter=1000,random_state=42)
Elas=LogisticRegression(penalty='elasticnet',solver='saga',l1_ratio=0.5,C=1/0.1,max_iter=1000,random_state=42)
Sgd=SGDClassifier(loss='log_loss',max_iter=1000,alpha=0.001,random_state=42,eta0=0.01,learning_rate='optimal')
svr=SVC(max_iter=100,gamma=0.1,kernel='rbf',C=0.6,degree=3,probability=True,random_state=42)
Rg=RandomForestClassifier(n_estimators=100,criterion='gini',min_samples_leaf=2,min_samples_split=3,max_features='sqrt',random_state=42)
Gb=GradientBoostingClassifier(n_estimators=100,subsample=1,min_samples_leaf=3,max_features='sqrt',random_state=42)
Ada=AdaBoostClassifier(n_estimators=156,random_state=42)
Dtr=DecisionTreeClassifier(max_depth=3,min_samples_leaf=3,min_samples_split=3,random_state=42,criterion='gini')
Knr=KNeighborsClassifier(n_neighbors=17,weights='distance',metric='minkowski')
XG=XGBClassifier(n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1)
cat_boost=CatBoostClassifier(verbose=0)
lightgbm=LGBMClassifier()


stacking=StackingClassifier(
    estimators=[
         ('lr',lr),
         ('elas',Elas),
         ('Sgd',Sgd),
         ("svr",svr),
         ('Rg',Rg),
         ("GB",Gb),
         ("Ada",Ada),
         ("Dtr",Dtr),
         ('Knr',Knr),
         ('XG',XG),
         ('cat_boost',cat_boost),
         ('lightgbm',lightgbm)

    ],
   final_estimator=LogisticRegression(max_iter=1000)
)


voting=VotingClassifier(
    estimators=[
         ('lr',lr),
         ('elas',Elas),
         ('Sgd',Sgd),
         ("svr",svr),
         ('Rg',Rg),
         ("GB",Gb),
         ("Ada",Ada),
         ("Dtr",Dtr),
         ('Knr',Knr),
         ('XG',XG),
         ('cat_boost',cat_boost),
         ('lightgbm',lightgbm)

    ],
    voting='soft'

)


model_to_train={
    'Logistic Regression' : lr,
    'Elas':Elas,
    "SGDClassifier":Sgd,
    "SVC":svr,
    "RandomForestClassifier":Rg,
    "GradientBoostingClassifier":Gb,
    "AdaBoostClassifier":Ada,
    'DecisionTreeClassifier':Dtr,
    "KNeighborsClassifier":Knr,
    "XGBClassifier":XG,
    "StackingClassifier":stacking,
    "VotingClassifier":voting,
    "CatBoostClassifier":cat_boost,
    "LGBMClassifier":lightgbm


}


from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)


# Trainning pipe line (SMOTE add kora hoyeche imbalance class handle korar jonno)
result=[]
for model_name,model in model_to_train.items():
    pipe=Pipeline(
        [("preprocessor",preprocessor),
        ("smote",SMOTE(random_state=42)),
        ("model",model)
        ]
    )
    pipe.fit(x_train,y_train)
    y_pred=pipe.predict(x_test)
    accuracy=accuracy_score(y_test,y_pred)
    result.append(
        {
            "model":model,
            'accuracy':accuracy
        }
    )


res=pd.DataFrame(result).sort_values(by='accuracy',ascending=False)
res

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
import numpy as np
import pandas as pd

# Training Pipeline (with SMOTE)
result = []

for model_name, model in model_to_train.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])

    pipe.fit(x_train, y_train)

    y_pred = pipe.predict(x_test)

    # Classification Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')

    result.append({
        "Model": model_name,
        "Accuracy": acc,
        "F1 Score": f1,
        "Precision": precision,
        "Recall": recall
    })

# Convert to DataFrame
res = pd.DataFrame(result).sort_values(
    by="Accuracy",
    ascending=False
)

res


xg_pipeline= Pipeline(
    [
        ('preprocessor',preprocessor),
        ('smote',SMOTE(random_state=42)),
        ('model',XG)
    ]
)



# Cross validation
cv_result=cross_val_score(xg_pipeline,x_train,y_train,cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42),scoring='accuracy')

cv_acc=cv_result.mean()
cv_std=cv_result.std()

cv_acc,cv_std,cv_result



param_grid = {
    "model__n_estimators": [ 900, 1000, 1100],
    "model__learning_rate": [0.04, 0.1, 0.2],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_child_weight": [1, 2, 3],
    "model__subsample": [0.4, 0.5, 0.8],
    "model__colsample_bytree": [0.5, 0.8, 1.0],
    "model__reg_alpha": [0.1, 0.2, 0.5],
    "model__reg_lambda": [1, 2, 4]
}

grid=GridSearchCV(
    estimator=xg_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1

)

grid.fit(x_train,y_train)
print(grid.best_params_)
print(grid.best_score_)

model = grid.best_estimator_

feature_names = model.named_steps["preprocessor"].get_feature_names_out()

importance = model.named_steps["model"].feature_importances_

importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance
    })
    .sort_values("Importance", ascending=False)
)

print(importance.head(30))

preds=model.predict(test_df)
# label encoder diye abar original class label e ferot newa hocche
preds=target_encoder.inverse_transform(preds)

submission=pd.Dataframe(
    {
        'PID':test_ids,
        TARGET_COL:preds
    }
)
submission.to_csv('submission.csv',index=False)